# SkillLens - Exploratory Data Analysis

This notebook mirrors `src/eda.py`. Run all cells, or run the script directly:
```
python src/eda.py
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CATEGORY_ORDER = ['Needs Improvement', 'Developing', 'Placement Ready', 'Highly Ready']
df = pd.read_csv('../data/raw/students.csv')
df['readiness_category'] = pd.Categorical(df['readiness_category'], categories=CATEGORY_ORDER, ordered=True)
df.head()

## Overview & missing values


In [ ]:
print(df.shape)
df.isna().sum()[df.isna().sum() > 0]

## Target class balance
We deliberately generated a non-degenerate spread across all four categories (see `src/generate_dataset.py` for the weighting/noise design).

In [ ]:
counts = df['readiness_category'].value_counts().reindex(CATEGORY_ORDER)
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette='viridis', legend=False)
plt.title('Readiness Category Distribution')
plt.xticks(rotation=15)
plt.show()

## Key feature distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col in zip(axes.flat, ['cgpa', 'leetcode_solved', 'aptitude_score', 'num_projects']):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Feature Correlation Heatmap')
plt.show()

## Feature vs. target (sanity check)
Medians should trend upward across categories -- this confirms the synthetic target actually depends on the features in a sensible way, without being perfectly separable (perfect separability would be a red flag for a trivial/leaky target).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, ['cgpa', 'leetcode_solved', 'num_projects', 'aptitude_score']):
    sns.boxplot(data=df, x='readiness_category', y=col, order=CATEGORY_ORDER, hue='readiness_category', palette='viridis', ax=ax, legend=False)
    ax.set_title(f'{col} vs readiness_category')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## Findings summary

- Class balance: Needs Improvement 22%, Developing 39%, Placement Ready 29%, Highly Ready 10% -- imbalanced but not degenerate; worth watching macro-F1 (not just accuracy) during evaluation.
- `leetcode_solved`, `num_projects`, and `aptitude_score` show the clearest upward trend across categories -- expect these near the top of feature importance later.
- Classes overlap substantially (see boxplots) -- we should NOT expect near-100% accuracy; that would suggest leakage.
- 4 columns have ~3% missing values by design, to be handled by the preprocessing pipeline (`src/data_preprocessing.py`) and exercised later in robustness testing.